# 24 — Post processing

Adds raw data of _apparent_temp_monthly.nc, _precipitation_monthly.nc and _population_density.nc

In [3]:
import geopandas as gpd
import httpx
import pandas as pd
import xarray as xr

from common import PROCESSED_DIR, RAW_DIR

variable_raw = RAW_DIR / 'post_processing'
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Raw scoring inputs

Compact land-only parquet holding the three raw layers that `99_atlas_map.ipynb` re-scores on the fly under different climate / rainfall / density preferences: monthly apparent temperature, monthly precipitation, and population density. Ocean cells and rows outside every country polygon are dropped so the file is small enough to load quickly instead of pulling ~25 MB of gridded NetCDFs.

Layout is wide — one row per `(lat, lon)`, with 12 columns per monthly variable (`apparent_temp_01`..`apparent_temp_12`, `precipitation_01`..`precipitation_12`) and a scalar `population_density`. `common.load_raw_scoring_inputs` reconstructs the three DataArrays on the shared atlas grid.

In [ ]:
def _monthly_to_wide(da, prefix):
    """Unstack a ``(lat, lon, month)`` DataArray into a ``{prefix}_MM`` wide frame."""
    return (
        da.to_dataframe(name=prefix)
        .unstack('month')[prefix]
        .rename(columns=lambda m: f'{prefix}_{int(m):02d}')
    )

at_da = xr.open_dataarray(PROCESSED_DIR / '_apparent_temp_monthly.nc')
precip_da = xr.open_dataarray(PROCESSED_DIR / '_precipitation_monthly.nc')
density_da = xr.open_dataarray(PROCESSED_DIR / '_population_density.nc')

raw = (
    _monthly_to_wide(at_da, 'apparent_temp')
    .join(_monthly_to_wide(precip_da, 'precipitation'), how='outer')
    .join(density_da.to_dataframe(name='population_density'), how='outer')
    .reset_index()
)

value_cols = [c for c in raw.columns if c not in ('lat', 'lon')]
raw = raw.dropna(subset=value_cols, how='all').reset_index(drop=True)

out = PROCESSED_DIR / 'raw_scoring_inputs.parquet'
raw.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(raw):,} rows)')
raw.head()

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/raw_scoring_inputs.parquet (6.1 MB, 61,844 rows)


,lat,lon,apparent_temp_01,apparent_temp_02,apparent_temp_03,apparent_temp_04,apparent_temp_05,apparent_temp_06,apparent_temp_07,apparent_temp_08,...,precipitation_04,precipitation_05,precipitation_06,precipitation_07,precipitation_08,precipitation_09,precipitation_10,precipitation_11,precipitation_12,population_density
0,-55.25,-69.75,5.629470,5.833051,5.093107,3.276829,1.521110,-0.337384,-0.859134,-0.517225,...,137.669998,111.072998,118.980003,108.127998,104.718002,99.180000,110.608002,137.309998,140.306000,0.229051
1,-55.25,-69.25,5.420443,5.578864,4.708213,2.709335,0.796379,-1.179071,-1.709573,-1.258287,...,126.120003,102.455002,109.680000,97.774002,94.487999,89.430000,98.517998,124.199997,130.695999,0.125773
2,-55.25,-68.75,5.866284,5.972975,4.909866,2.673268,0.591875,-1.506476,-2.029464,-1.456782,...,90.000000,72.074997,78.599998,67.889999,63.240002,60.150002,63.549999,85.800003,94.860001,0.054949
3,-55.25,-68.25,6.272837,6.383113,5.288891,3.034797,1.000118,-1.042659,-1.570252,-1.059688,...,85.800003,68.478996,73.800003,64.293999,58.900002,56.549999,59.209999,80.279999,90.643997,0.011995
4,-55.25,-67.75,6.643222,6.772828,5.710204,3.489270,1.542122,-0.446346,-0.968214,-0.525966,...,81.959999,66.371002,70.620003,61.721001,56.884998,54.029999,56.264999,75.570000,85.591003,0.009441
